# 01. Dataset Preparation & Integrity Pipeline
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com)
### Academic Research Project: **GeoFinBERT**
Adapting `ProsusAI/FinBERT` to real-world geopolitical and energy-market news (2000–11 November 2026).

---
### Research Domain:
- **Oil and crude oil markets** (Brent, WTI, OPEC quotas, SPR, refining disruptions)
- **Gold and bullion markets** (Bullion reserves, safe-haven flows, inflation hedge)
- **US–Iran relations** (Sanctions, JCPOA, maritime seizures, diplomatic escalation)
- **Middle Eastern geopolitical events** (Strait of Hormuz, Persian Gulf security, regional conflicts)

---
### Strict Academic Data Integrity Principles:
1. **Zero Synthetic Data**: Never generate synthetic articles or artificial human labels.
2. **Zero Model Predictions as Ground Truth**: Never use LLM or model outputs as labels.
3. **URL and Date Preservation**: Retain canonical source URLs, publication timestamps, and accredited publishers.
4. **Wire Syndication Deduplication**: Detect and remove exact duplicates and near-duplicate syndicated stories.
5. **Zero Lookahead Leakage**: Strictly chronological train, validation, and test partitions.


In [ ]:
# Environment Setup & Automatic Colab Diagnostics
import os
import sys

try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    print("[Colab Environment Detected] Installing dependencies...")
    !pip install -q -r requirements.txt
    !pip install -q -e .

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sys.path.append(os.path.abspath("src"))

from geofinbert.config import load_config
from geofinbert.utils import set_seed, get_device_info
from geofinbert.data.fetcher import fetch_unlabeled_geofin_corpus, fetch_human_labeled_sentiment_corpus
from geofinbert.data.preprocessor import validate_schema_and_integrity, create_chronological_splits

config = load_config("configs/config.yaml")
set_seed(config["project"]["seed"])

device_info = get_device_info()
print("Hardware Check:", device_info)
if device_info["cuda_available"]:
    print(f"Accelerated GPU Active: {device_info['gpu_name']} ({device_info['vram_gb']} GB VRAM)")
else:
    print("Running in CPU mode. In Colab, enable GPU: Runtime -> Change runtime type -> T4/A100 GPU.")


## Step 1: Ingesting Genuine Historical Pretraining Corpus (DAPT)
We download authentic news articles across accredited financial and commodity publishers (ForexLive, Benzinga, ZeroHedge, OilPrice, MarketWatch, Economic Times, MoneyControl, Reuters syndication).


In [ ]:
# Fetch Unlabeled Geopolitical and Energy Market News Corpus
dapt_df = fetch_unlabeled_geofin_corpus(
    cache_dir=config["data"]["raw_dir"],
    processed_dir=config["data"]["processed_dir"]
)

print(f"Total unlabelled DAPT articles ingested: {len(dapt_df):,}")
dapt_df.head(3)


## Step 2: Ingesting Independently Human-Annotated Sentiment Corpus
We acquire real articles with independently human-annotated sentiment (`positive`, `negative`, `neutral`) using consensus financial annotations and commodity price-sentiment ground truth.


In [ ]:
# Fetch Human-Labeled Corpus
labeled_df = fetch_human_labeled_sentiment_corpus(
    cache_dir=config["data"]["raw_dir"],
    processed_dir=config["data"]["processed_dir"]
)

print(f"Total human-labeled sentiment articles ingested: {len(labeled_df):,}")
labeled_df.head(3)


## Step 3: Strict Schema Validation & Data Integrity Auditing
We verify that no synthetic artifacts exist, missing text is eliminated, and dates lie strictly between `2000-01-01` and `2026-11-11`.


In [ ]:
dapt_audit = validate_schema_and_integrity(dapt_df, is_labeled=False)
print("=== DAPT Integrity Audit ===")
print("Valid:", dapt_audit["valid"])
print("Stats:", dapt_audit["stats"])
if dapt_audit["errors"]:
    print("Errors:", dapt_audit["errors"])

labeled_audit = validate_schema_and_integrity(labeled_df, is_labeled=True)
print("\n=== Human-Labeled Sentiment Integrity Audit ===")
print("Valid:", labeled_audit["valid"])
print("Label Distribution:", labeled_audit["stats"].get("label_distribution"))
if labeled_audit["errors"]:
    print("Errors:", labeled_audit["errors"])


## Step 4: Chronological Partitioning (Zero Temporal Leakage)
To accurately simulate out-of-time forecasting and prevent lookahead bias:
$$\max(\text{Train Date}) \le \min(\text{Val Date}) \le \max(\text{Val Date}) \le \min(\text{Test Date})$$
We also verify cross-partition duplicate exclusion.


In [ ]:
splits_dir = config["data"]["splits_dir"]
os.makedirs(splits_dir, exist_ok=True)

# Split DAPT Corpus (85% Train, 15% Val)
dapt_train, dapt_val, _ = create_chronological_splits(
    dapt_df, date_col="date", train_ratio=0.85, val_ratio=0.15, test_ratio=0.0
)
dapt_train.to_parquet(os.path.join(splits_dir, "dapt_train.parquet"), index=False)
dapt_val.to_parquet(os.path.join(splits_dir, "dapt_val.parquet"), index=False)

# Split Human-Labeled Corpus (70% Train, 15% Val, 15% Test)
split_cfg = config["data"]["splits"]
l_train, l_val, l_test = create_chronological_splits(
    labeled_df,
    date_col="date",
    train_ratio=split_cfg["train_ratio"],
    val_ratio=split_cfg["val_ratio"],
    test_ratio=split_cfg["test_ratio"]
)
l_train.to_parquet(os.path.join(splits_dir, "sentiment_train.parquet"), index=False)
l_val.to_parquet(os.path.join(splits_dir, "sentiment_val.parquet"), index=False)
l_test.to_parquet(os.path.join(splits_dir, "sentiment_test.parquet"), index=False)

print(f"DAPT Splits: Train={len(dapt_train):,}, Val={len(dapt_val):,}")
print(f"Sentiment Splits: Train={len(l_train):,}, Val={len(l_val):,}, Test={len(l_test):,}")


## Step 5: Visualizing Dataset Statistics by Year, Topic, and Label


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# 1. Topic Breakdown
topic_counts = dapt_df["topic"].value_counts()
sns.barplot(x=topic_counts.values, y=topic_counts.index, ax=axes[0], palette="viridis")
axes[0].set_title("DAPT Articles by Research Topic")
axes[0].set_xlabel("Article Count")

# 2. Label Distribution
label_names = ["Positive (0)", "Negative (1)", "Neutral (2)"]
label_counts = [labeled_df["label"].value_counts().get(i, 0) for i in [0, 1, 2]]
sns.barplot(x=label_names, y=label_counts, ax=axes[1], palette="magma")
axes[1].set_title("Human Sentiment Ground Truth Distribution")
axes[1].set_ylabel("Count")

# 3. Timeline Coverage
dapt_years = pd.to_datetime(dapt_df["date"]).dt.year.value_counts().sort_index()
axes[2].plot(dapt_years.index, dapt_years.values, marker="o", color="navy")
axes[2].set_title("Corpus Timeline (2000–2026)")
axes[2].set_xlabel("Year")
axes[2].set_ylabel("Article Count")
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


## Step 6: Dataset Summary Verification
The cleaned, verified datasets are now serialized in `data/splits/` ready for Stage 1 DAPT pretraining and Stage 2 Fine-Tuning.
